## **Step 1**: Install Spark

## **Step 2**: Setting up Spark

This Colab-ready notebook follows the same **HW practice** flow, but it is updated to work directly with your uploaded `airline.csv` file in Google Colab.


## **Step 3**: Import the libraries and create a Spark session

In [1]:

# Colab-friendly setup for PySpark
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null
!pip install -q "pyspark[connect]==3.5.1" "dataproc-spark-connect==0.8.3"

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws, lpad, avg, min as spark_min, max as spark_max

spark = SparkSession.builder.master("local[*]").appName("HWPracticeAirline").getOrCreate()
sc = spark.sparkContext

print("Spark version:", spark.version)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 6.9 MB/s eta 0:00:00
Spark version: 3.5.1


## **Step 4**: Upload and read the `airline.csv` file

Run the next cell and upload `airline.csv` from your computer into Colab.


In [3]:
## mount to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:

# Read the airline.csv file from Colab local storage
df = spark.read.option("header", "true").option("inferSchema", "true").csv("/content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Assignments/Dataset/airline.csv")
df.show(20, truncate=False)


+----+-----+----------+---------+-------+----------+-------+----------+-------------+---------+-------+-----------------+--------------+-------+--------+--------+------+----+--------+------+-------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|Year|Month|DayofMonth|DayOfWeek|DepTime|CRSDepTime|ArrTime|CRSArrTime|UniqueCarrier|FlightNum|TailNum|ActualElapsedTime|CRSElapsedTime|AirTime|ArrDelay|DepDelay|Origin|Dest|Distance|TaxiIn|TaxiOut|Cancelled|CancellationCode|Diverted|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|
+----+-----+----------+---------+-------+----------+-------+----------+-------------+---------+-------+-----------------+--------------+-------+--------+--------+------+----+--------+------+-------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|2005|1    |1         |6        |1211   |1216      |1451   |1502      |UA           |548      

Output the column names and the "rows" and "column" counts

In [6]:
df.columns  # Column Names

['Year',
 'Month',
 'DayofMonth',
 'DayOfWeek',
 'DepTime',
 'CRSDepTime',
 'ArrTime',
 'CRSArrTime',
 'UniqueCarrier',
 'FlightNum',
 'TailNum',
 'ActualElapsedTime',
 'CRSElapsedTime',
 'AirTime',
 'ArrDelay',
 'DepDelay',
 'Origin',
 'Dest',
 'Distance',
 'TaxiIn',
 'TaxiOut',
 'Cancelled',
 'CancellationCode',
 'Diverted',
 'CarrierDelay',
 'WeatherDelay',
 'NASDelay',
 'SecurityDelay',
 'LateAircraftDelay']

In [7]:

print("Row Count:", df.count())
print("Column Count:", len(df.columns))


Row Count: 539895
Column Count: 29


In [8]:
df.dtypes

[('Year', 'int'),
 ('Month', 'int'),
 ('DayofMonth', 'int'),
 ('DayOfWeek', 'int'),
 ('DepTime', 'string'),
 ('CRSDepTime', 'int'),
 ('ArrTime', 'string'),
 ('CRSArrTime', 'int'),
 ('UniqueCarrier', 'string'),
 ('FlightNum', 'int'),
 ('TailNum', 'string'),
 ('ActualElapsedTime', 'string'),
 ('CRSElapsedTime', 'string'),
 ('AirTime', 'string'),
 ('ArrDelay', 'string'),
 ('DepDelay', 'string'),
 ('Origin', 'string'),
 ('Dest', 'string'),
 ('Distance', 'int'),
 ('TaxiIn', 'string'),
 ('TaxiOut', 'string'),
 ('Cancelled', 'int'),
 ('CancellationCode', 'string'),
 ('Diverted', 'int'),
 ('CarrierDelay', 'string'),
 ('WeatherDelay', 'string'),
 ('NASDelay', 'string'),
 ('SecurityDelay', 'string'),
 ('LateAircraftDelay', 'string')]

In order to examine the summary of any particular column of a DataFrame, we use the `describe` method.

The `describe` method gives us the statistical summary of the given column.


In [9]:
df.describe('DepDelay').show()

+-------+-----------------+
|summary|         DepDelay|
+-------+-----------------+
|  count|           539895|
|   mean|11.16137993946179|
| stddev|34.37881570185133|
|    min|               -1|
|    max|               NA|
+-------+-----------------+



In order to select particular columns from the DataFrame, use the `select` method

In [10]:
df.select('ArrDelay', 'DepDelay').show()

+--------+--------+
|ArrDelay|DepDelay|
+--------+--------+
|     -11|      -5|
|     -15|      -7|
|      -8|      -3|
|      NA|      NA|
|       2|      -5|
|       1|      -1|
|      75|      75|
|     -17|      -2|
|     -22|      -9|
|      65|      83|
|     -16|      -4|
|     -22|      -7|
|     -23|      -5|
|     -14|       3|
|     -17|      -6|
|     -16|      -4|
|     -15|      -6|
|     -13|      -4|
|     -10|      -5|
|      -1|       2|
+--------+--------+
only showing top 20 rows



Selecting distinct multiple columns

In [11]:
df.select('ArrDelay', 'DepDelay').distinct().show()

+--------+--------+
|ArrDelay|DepDelay|
+--------+--------+
|     -23|      -5|
|     -20|      -6|
|       5|      10|
|      58|      62|
|      41|      31|
|      71|      77|
|      37|      45|
|       6|      -9|
|      65|      73|
|     102|      83|
|      56|      33|
|     123|     140|
|      80|      33|
|      55|      41|
|      69|      77|
|     111|     100|
|     701|     694|
|     101|     100|
|     136|     131|
|     112|      97|
+--------+--------+
only showing top 20 rows



## **Step 5**: Filtering Data

In [12]:
df.filter(df.Origin == 'SFO').show()

+----+-----+----------+---------+-------+----------+-------+----------+-------------+---------+-------+-----------------+--------------+-------+--------+--------+------+----+--------+------+-------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|Year|Month|DayofMonth|DayOfWeek|DepTime|CRSDepTime|ArrTime|CRSArrTime|UniqueCarrier|FlightNum|TailNum|ActualElapsedTime|CRSElapsedTime|AirTime|ArrDelay|DepDelay|Origin|Dest|Distance|TaxiIn|TaxiOut|Cancelled|CancellationCode|Diverted|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|
+----+-----+----------+---------+-------+----------+-------+----------+-------------+---------+-------+-----------------+--------------+-------+--------+--------+------+----+--------+------+-------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|2005|    1|         1|        6|   1211|      1216|   1451|      1502|           UA|      548

Filtering data with multiple parameters

In [13]:
df.filter((df.Origin == 'SFO') & (df.Dest == 'PDX')).show()

+----+-----+----------+---------+-------+----------+-------+----------+-------------+---------+-------+-----------------+--------------+-------+--------+--------+------+----+--------+------+-------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|Year|Month|DayofMonth|DayOfWeek|DepTime|CRSDepTime|ArrTime|CRSArrTime|UniqueCarrier|FlightNum|TailNum|ActualElapsedTime|CRSElapsedTime|AirTime|ArrDelay|DepDelay|Origin|Dest|Distance|TaxiIn|TaxiOut|Cancelled|CancellationCode|Diverted|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|
+----+-----+----------+---------+-------+----------+-------+----------+-------------+---------+-------+-----------------+--------------+-------+--------+--------+------+----+--------+------+-------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|2005|    1|         6|        4|   1830|      1831|   2005|      2012|           UA|      558

## **Step 6**: Performing SQL Queries

SQL queries can be passed directly to a DataFrame after creating a temporary view.


In [14]:

# Create a temporary SQL view
df.createOrReplaceTempView("airlines")
print("Temp view 'airlines' created.")


Temp view 'airlines' created.


In [15]:

from pyspark.sql import SQLContext
sqlContext = SQLContext(spark)
sqlContext


/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [16]:
sqlContext.sql('select * from airlines').show()

+----+-----+----------+---------+-------+----------+-------+----------+-------------+---------+-------+-----------------+--------------+-------+--------+--------+------+----+--------+------+-------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|Year|Month|DayofMonth|DayOfWeek|DepTime|CRSDepTime|ArrTime|CRSArrTime|UniqueCarrier|FlightNum|TailNum|ActualElapsedTime|CRSElapsedTime|AirTime|ArrDelay|DepDelay|Origin|Dest|Distance|TaxiIn|TaxiOut|Cancelled|CancellationCode|Diverted|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|
+----+-----+----------+---------+-------+----------+-------+----------+-------------+---------+-------+-----------------+--------------+-------+--------+--------+------+----+--------+------+-------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+
|2005|    1|         1|        6|   1211|      1216|   1451|      1502|           UA|      548

In [18]:
sqlContext.sql('select distinct(Dest) from airlines').show(20)

+----+
|Dest|
+----+
| MSY|
| SNA|
| BUR|
| IDA|
| EUG|
| RDM|
| MOD|
| CEC|
| LIH|
| IAH|
| HNL|
| CVG|
| SJC|
| RDD|
| AUS|
| GJT|
| BFL|
| RNO|
| BOS|
| EWR|
+----+
only showing top 20 rows



In [19]:
print(df.count())

539895


In [20]:
print(df.take(3))

[Row(Year=2005, Month=1, DayofMonth=1, DayOfWeek=6, DepTime='1211', CRSDepTime=1216, ArrTime='1451', CRSArrTime=1502, UniqueCarrier='UA', FlightNum=548, TailNum='N341UA', ActualElapsedTime='100', CRSElapsedTime='106', AirTime='81', ArrDelay='-11', DepDelay='-5', Origin='SFO', Dest='SLC', Distance=599, TaxiIn='2', TaxiOut='17', Cancelled=0, CancellationCode=None, Diverted=0, CarrierDelay='0', WeatherDelay='0', NASDelay='0', SecurityDelay='0', LateAircraftDelay='0'), Row(Year=2005, Month=1, DayofMonth=2, DayOfWeek=7, DepTime='1209', CRSDepTime=1216, ArrTime='1447', CRSArrTime=1502, UniqueCarrier='UA', FlightNum=548, TailNum='N398UA', ActualElapsedTime='98', CRSElapsedTime='106', AirTime='79', ArrDelay='-15', DepDelay='-7', Origin='SFO', Dest='SLC', Distance=599, TaxiIn='2', TaxiOut='17', Cancelled=0, CancellationCode=None, Diverted=0, CarrierDelay='0', WeatherDelay='0', NASDelay='0', SecurityDelay='0', LateAircraftDelay='0'), Row(Year=2005, Month=1, DayofMonth=3, DayOfWeek=1, DepTime='12

In [21]:

spark.sql("""
SELECT DISTINCT Origin, Dest, Distance
FROM airlines
WHERE Distance > 1000
ORDER BY Distance DESC
""").show(10)


+------+----+--------+
|Origin|Dest|Distance|
+------+----+--------+
|   SFO| BOS|    2704|
|   SFO| JFK|    2586|
|   SFO| MIA|    2585|
|   SFO| EWR|    2565|
|   SFO| PHL|    2521|
|   SFO| BWI|    2457|
|   SFO| LIH|    2447|
|   SFO| MCO|    2445|
|   SFO| IAD|    2419|
|   SFO| HNL|    2398|
+------+----+--------+
only showing top 10 rows



## **Step 7**: Aggregation

In [22]:

spark.sql("""
SELECT concat(Origin, Dest) as market,
       avg(Distance) as avg_distance
FROM airlines
GROUP BY 1
ORDER BY 1 DESC
""").show(10)


+------+------------+
|market|avg_distance|
+------+------------+
|SFOTWF|       536.0|
|SFOTUS|       751.0|
|SFOSTL|      1736.0|
|SFOSNA|       372.0|
|SFOSMX|       216.0|
|SFOSMF|        86.0|
|SFOSLC|       599.0|
|SFOSJC|        30.0|
|SFOSEA|       679.0|
|SFOSBP|       191.0|
+------+------------+
only showing top 10 rows



In [23]:

spark.sql("""
SELECT concat(Origin, Dest) as market,
       min(Distance) as min_distance
FROM airlines
GROUP BY 1
ORDER BY 1 DESC
""").show(10)


+------+------------+
|market|min_distance|
+------+------------+
|SFOTWF|         536|
|SFOTUS|         751|
|SFOSTL|        1736|
|SFOSNA|         372|
|SFOSMX|         216|
|SFOSMF|          86|
|SFOSLC|         599|
|SFOSJC|          30|
|SFOSEA|         679|
|SFOSBP|         191|
+------+------------+
only showing top 10 rows



In [24]:

spark.sql("""
SELECT concat(Origin, Dest) as market,
       max(Distance) as max_distance
FROM airlines
GROUP BY 1
ORDER BY max_distance DESC
LIMIT 10
""").show(10)


+------+------------+
|market|max_distance|
+------+------------+
|SFOBOS|        2704|
|SFOJFK|        2586|
|SFOMIA|        2585|
|SFOEWR|        2565|
|SFOPHL|        2521|
|SFOBWI|        2457|
|SFOLIH|        2447|
|SFOMCO|        2445|
|SFOIAD|        2419|
|SFOHNL|        2398|
+------+------------+



## **Step 8**: Practice aligned with the earlier notebook style

The original updated notebook also used a simplified practice table with:
- `date`
- `delay`
- `origin`
- `destination`

This step creates those columns from your real airline dataset so you can practice Spark SQL in that style too.


In [25]:

practice_df = (
    df
    .withColumn(
        "date",
        concat_ws(
            "-",
            col("Year").cast("string"),
            lpad(col("Month").cast("string"), 2, "0"),
            lpad(col("DayofMonth").cast("string"), 2, "0")
        )
    )
    .withColumnRenamed("DepDelay", "delay")
    .withColumnRenamed("Origin", "origin")
    .withColumnRenamed("Dest", "destination")
)

practice_df.select("date", "delay", "origin", "destination", "UniqueCarrier", "FlightNum").show(10, truncate=False)


+----------+-----+------+-----------+-------------+---------+
|date      |delay|origin|destination|UniqueCarrier|FlightNum|
+----------+-----+------+-----------+-------------+---------+
|2005-01-01|-5   |SFO   |SLC        |UA           |548      |
|2005-01-02|-7   |SFO   |SLC        |UA           |548      |
|2005-01-03|-3   |SFO   |SLC        |UA           |548      |
|2005-01-04|NA   |SFO   |SLC        |UA           |548      |
|2005-01-05|-5   |SFO   |SLC        |UA           |548      |
|2005-01-06|-1   |SFO   |SLC        |UA           |548      |
|2005-01-07|75   |SFO   |SLC        |UA           |548      |
|2005-01-08|-2   |SFO   |SLC        |UA           |548      |
|2005-01-09|-9   |SFO   |SLC        |UA           |548      |
|2005-01-10|83   |SFO   |SLC        |UA           |548      |
+----------+-----+------+-----------+-------------+---------+
only showing top 10 rows



In [26]:

practice_df.createOrReplaceTempView("us_delay_flights_tbl")

print(spark.catalog.listDatabases())
print(spark.catalog.listTables())


[Database(name='default', catalog='spark_catalog', description='default database', locationUri='file:/content/spark-warehouse')]
[Table(name='airlines', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True), Table(name='us_delay_flights_tbl', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]


In [27]:

df_sfo = spark.sql("""
SELECT date, delay, origin, destination, UniqueCarrier, FlightNum
FROM us_delay_flights_tbl
WHERE origin = 'SFO'
ORDER BY date
""")

df_jfk = spark.sql("""
SELECT date, delay, origin, destination, UniqueCarrier, FlightNum
FROM us_delay_flights_tbl
WHERE origin = 'JFK'
ORDER BY date
""")

print("SFO flights:")
df_sfo.show(10, truncate=False)

print("JFK flights:")
df_jfk.show(10, truncate=False)


SFO flights:
+----------+-----+------+-----------+-------------+---------+
|date      |delay|origin|destination|UniqueCarrier|FlightNum|
+----------+-----+------+-----------+-------------+---------+
|2005-01-01|-3   |SFO   |ORD        |UA           |696      |
|2005-01-01|0    |SFO   |DEN        |UA           |728      |
|2005-01-01|-5   |SFO   |SLC        |UA           |548      |
|2005-01-01|72   |SFO   |DEN        |UA           |662      |
|2005-01-01|-4   |SFO   |SNA        |UA           |671      |
|2005-01-01|-2   |SFO   |SNA        |UA           |661      |
|2005-01-01|41   |SFO   |PDX        |UA           |580      |
|2005-01-01|11   |SFO   |PDX        |UA           |704      |
|2005-01-01|11   |SFO   |SNA        |UA           |709      |
|2005-01-01|-2   |SFO   |RNO        |UA           |710      |
+----------+-----+------+-----------+-------------+---------+
only showing top 10 rows

JFK flights:
+----+-----+------+-----------+-------------+---------+
|date|delay|origin|desti

In [28]:

df_jfk.createOrReplaceTempView("us_origin_airport_JFK_tmp_view")

spark.sql("""
SELECT *
FROM us_origin_airport_JFK_tmp_view
LIMIT 10
""").show(truncate=False)


+----+-----+------+-----------+-------------+---------+
|date|delay|origin|destination|UniqueCarrier|FlightNum|
+----+-----+------+-----------+-------------+---------+
+----+-----+------+-----------+-------------+---------+



In [29]:

spark.sql("""
SELECT origin, COUNT(*) as total_flights, AVG(delay) as avg_departure_delay
FROM us_delay_flights_tbl
GROUP BY origin
ORDER BY total_flights DESC
LIMIT 10
""").show()


+------+-------------+-------------------+
|origin|total_flights|avg_departure_delay|
+------+-------------+-------------------+
|   SFO|       539895|  11.16137993946179|
+------+-------------+-------------------+

